<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

Cloning into 'ML_fx'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 78 (delta 28), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 20.90 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())

   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
1    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
2    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
3    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
4    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

In [3]:
!pip install wandb -q
!pip install neuralforecast torch pytorch-lightning
!pip install pytorch-forecasting pandas numpy torch matplotlib


import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [10]:
from prophet import Prophet
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import time

# 1. Identify unique Store/Dept combinations
combinations = train.groupby(['Store', 'Dept']).size().reset_index(name='count')
combos_to_fit = combinations.head(1000).reset_index(drop=True)

print(f"Fitting Prophet for {len(combos_to_fit)} series...")

def fit_prophet(store_id, dept_id):
    try:
        # Filter data
        df = train[(train['Store'] == store_id) & (train['Dept'] == dept_id)].copy()
        if len(df) < 20:
            return None

        df = df.sort_values('Date').reset_index(drop=True)

        # Split into train and val based on split_date
        val_start_date = '2011-12-01'
        train_df = df[df['Date'] <= val_start_date]
        val_df = df[df['Date'] > val_start_date]

        if len(val_df) == 0:
            return None

        # Rename for Prophet
        prophet_train = train_df.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
        prophet_val = val_df.rename(columns={'Date': 'ds'})

        # Fit Model
        model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
        model.fit(prophet_train)

        # Predict
        forecast = model.predict(prophet_val)

        # Get actuals
        val_actuals = val_df['Weekly_Sales'].values
        val_preds = forecast['yhat'].values

        mae = np.mean(np.abs(val_actuals - val_preds))
        return {'Store': store_id, 'Dept': dept_id, 'MAE': mae}

    except Exception as e:
        pass
    return None

# 2. Run in Parallel using all CPU cores
start_time = time.time()
results = Parallel(n_jobs=-1)(delayed(fit_prophet)(row['Store'], row['Dept']) for _, row in combos_to_fit.iterrows())

end_time = time.time()
print(f"Finished in {end_time - start_time:.2f} seconds")

# 3. Aggregate Results
results_df = pd.DataFrame([r for r in results if r is not None])
if not results_df.empty:
    avg_mae = results_df['MAE'].mean()
    print(f"Average MAE over {len(results_df)} series: {avg_mae:.2f}")
else:
    print("No results generated.")


Fitting Prophet for 1000 series...
Finished in 544.57 seconds
Average MAE over 939 series: 2551.62
